# ML test — real-data fire model
Retrains the three CFI-G models on the **175 usable real ISS tests** (`Data/raw/psi159_experiments.json`), validated leave-one-test-day-out, then checks predictions.

Outputs: `predicted_end_mode` (no_ignition | self_extinguished | flow_terminated), `hot_ignition`, `burn_time_s`, `hazard_index`, `level`. Cool flames are **not** predicted (only 2 test days examined them).

In [1]:
from pathlib import Path
import sys

# Works on every machine: find the project root (the folder that contains rag/pipeline.py)
PROJECT_ROOT = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "rag" / "pipeline.py").exists()
)
RAG_DIR = PROJECT_ROOT / "rag"
if str(RAG_DIR) not in sys.path:
    sys.path.insert(0, str(RAG_DIR))

for name in ["artifacts/fire_model.joblib", "artifacts/metrics.json", "predict.py", "train.py", "features.py", "scoring.py"]:
    path = RAG_DIR / "ml" / name
    print(f"{name}: exists={path.exists()}")

artifacts/fire_model.joblib: exists=True
artifacts/metrics.json: exists=True
predict.py: exists=True
train.py: exists=True
features.py: exists=True
scoring.py: exists=True


In [2]:
# Retrain on this machine (keeps the model file compatible with the installed scikit-learn).
import importlib
import ml.train as train_module
importlib.reload(train_module)

metrics = train_module.train()
print("\nLeave-one-test-day-out scores:")
for target in ("ignition", "sustained", "burn_time"):
    print(" ", target, metrics.get(target, {}).get("chosen", metrics.get(target, "")))

usable rows: 175 / 179  (ignited+terminated: 132, with burn time: 132)
ignition   random_forest  LODO ROC-AUC=0.77 acc=0.771 (baseline 0.754) | random-5fold ROC-AUC=0.891
sustained  logreg         LODO ROC-AUC=0.795 acc=0.75 (baseline 0.773) | random-5fold ROC-AUC=0.813
burn_time  random_forest  LODO MAE=29.6s (baseline 43.4s) | random-5fold MAE=25.4s
O2 sweep (pooled fuels, 1 atm):
  O2 18%  P(ignite)=0.73  P(sustained|lit)=0.01  burn~10s  hazard=13
  O2 21%  P(ignite)=0.79  P(sustained|lit)=0.03  burn~10s  hazard=15
  O2 25%  P(ignite)=0.83  P(sustained|lit)=0.07  burn~10s  hazard=17
  O2 30%  P(ignite)=0.87  P(sustained|lit)=0.20  burn~126s  hazard=44
  O2 35%  P(ignite)=0.87  P(sustained|lit)=0.44  burn~126s  hazard=50
  O2 40%  P(ignite)=0.94  P(sustained|lit)=0.72  burn~135s  hazard=63
saved -> ml\artifacts\fire_model.joblib , ml\artifacts\metrics.json

Leave-one-test-day-out scores:
  ignition random_forest
  sustained logreg
  burn_time random_forest


In [3]:
import ml.predict as predict
importlib.reload(predict)
predict._load.cache_clear()

print("Model path:", predict.MODEL_PATH)
print("Model exists:", predict.MODEL_PATH.exists())

Model path: C:\Users\Ahmed\.zcode\workspace\default\Nasa Rag SYS\rag\ml\artifacts\fire_model.joblib
Model exists: True


In [4]:
# Enriched-oxygen ISS test-chamber condition
enriched = predict.predict({
    "fuel": "30% butane",
    "oxygen_before_pct": 38,
    "pressure_kpa": 202.7,
    "fuel_flow_mg_s": 1.0,
})
print({k: enriched[k] for k in ("predicted_end_mode", "end_mode_probabilities", "burn_time_s", "hazard_index", "level")})
print("caveats:", enriched["caveats"])
assert enriched["predicted_end_mode"] in {"no_ignition", "self_extinguished", "flow_terminated"}
assert enriched["caveats"], "enriched O2 must carry the test-chamber caveat"

{'predicted_end_mode': 'flow_terminated', 'end_mode_probabilities': {'no_ignition': 0.0512, 'self_extinguished': 0.31, 'flow_terminated': 0.6388}, 'burn_time_s': 115.8, 'hazard_index': 59.9, 'level': 'HIGH'}
caveats: ['O2 levels above ~23% in these tests (33-40%) are ISS test-chamber conditions, not a spacecraft cabin atmosphere (ISS air is ~21%).']


In [5]:
# Normal ISS cabin air
cabin = predict.predict({
    "fuel": "propane",
    "oxygen_before_pct": 20.9,
    "pressure_kpa": 101.3,
})
print({k: cabin[k] for k in ("predicted_end_mode", "end_mode_probabilities", "burn_time_s", "hazard_index", "level")})
print(predict.model_info()["not_predicted"])

{'predicted_end_mode': 'self_extinguished', 'end_mode_probabilities': {'no_ignition': 0.3317, 'self_extinguished': 0.6087, 'flow_terminated': 0.0596}, 'burn_time_s': 8.8, 'hazard_index': 13.8, 'level': 'LOW'}
{'cool_flame': "only examined on test days 21174 and 21291 (9 positives) — an empty cell elsewhere means 'not examined', so no valid label", 'peak_burner_temp_c': 'measured during the burn (would leak the outcome)'}
